In [ ]:
import shutil, subprocess

print("GPU:")
print(subprocess.run(
    ["nvidia-smi"],
    capture_output=True, text=True
).stdout)

print("\nDisk space:")
for path in ["/tmp", "/kaggle/working"]:
    print(path, round(shutil.disk_usage(path).free / 1e9, 1), "GB free")

In [ ]:
import os, sys, json, time, shutil, subprocess
from pathlib import Path
os.environ["WANDB_DISABLED"] = "true"
t0 = time.time()

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "NO GPU - STOP, enable GPU in Session options")

URL = "https://www.seanoe.org/data/00858/96963/data/106157.tar.gz"
BASE = Path("/tmp/bepli"); RAW = BASE / "raw"; ARCH = BASE / "bepli.tar.gz"
YOLO_DIR = Path("/tmp/yolo_bepli"); OUTD = Path("/kaggle/working/runs"); OUTD.mkdir(parents=True, exist_ok=True)
EXPECTED = {"train": (2234, 71889), "val": (744, 24643), "test": (744, 22040)}
BASE.mkdir(parents=True, exist_ok=True); RAW.mkdir(parents=True, exist_ok=True)

def find_root():
    for p in RAW.rglob("plastic_coco"):
        if (p / "annotation").is_dir() and (p / "images").is_dir(): return p
    return None

# ---------- restore dataset ----------
ROOT = find_root()
if ROOT is None:
    free = shutil.disk_usage("/tmp").free / 1e9
    print(f"Free disk in /tmp: {free:.1f} GB")
    if free < 20: raise SystemExit("STOP: less than 20 GB free in /tmp. Share this message.")
    if not ARCH.exists() or ARCH.stat().st_size < 7e9:
        print("Downloading ...")
        rc = subprocess.call(["wget", "-q", "-c", "--tries=5", "--timeout=60", "-O", str(ARCH), URL])
        if rc != 0: raise SystemExit(f"STOP: download failed (wget exit {rc}). Share this message.")
    print(f"Archive: {ARCH.stat().st_size/1e9:.2f} GB | {time.time()-t0:.0f}s")
    if subprocess.call(["gzip", "-t", str(ARCH)]) != 0:
        raise SystemExit("STOP: archive corrupt. Delete /tmp/bepli/bepli.tar.gz and rerun.")
    print("Extracting ...")
    if subprocess.call(["tar", "-xzf", str(ARCH), "-C", str(RAW)]) != 0: raise SystemExit("STOP: extraction failed.")
    ROOT = find_root()
    if ROOT is None: raise SystemExit("STOP: plastic_coco folder not found after extraction.")
    os.remove(ARCH)
print("Dataset root:", ROOT)

# ---------- convert COCO -> YOLO (official splits unchanged) ----------
cats, cmap, counts = None, None, {}
for s, (ni, na) in EXPECTED.items():
    p = [q for q in sorted((ROOT / "annotation").glob("*.json")) if s in q.name.lower()][0]
    d = json.load(open(p))
    if len(d["images"]) != ni or len(d["annotations"]) != na: raise SystemExit(f"STOP: count mismatch in {s}")
    if cats is None:
        cats = sorted(d["categories"], key=lambda c: c["id"]); cmap = {c["id"]: i for i, c in enumerate(cats)}
    (YOLO_DIR / "images" / s).mkdir(parents=True, exist_ok=True); (YOLO_DIR / "labels" / s).mkdir(parents=True, exist_ok=True)
    by = {}
    for a in d["annotations"]: by.setdefault(a["image_id"], []).append(a)
    n_lines = 0
    for im in d["images"]:
        fn = Path(im["file_name"]).name
        src = ROOT / "images" / s / fn
        if not src.exists(): src = ROOT / "images" / s / im["file_name"]
        dst = YOLO_DIR / "images" / s / fn
        if not dst.exists():
            try: os.link(src, dst)
            except OSError: shutil.copy(src, dst)
        W, H = im["width"], im["height"]; lines = []
        for a in by.get(im["id"], []):
            x, y, w, h = a["bbox"]
            x0, y0, x1, y1 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
            if x1 <= x0 or y1 <= y0: continue
            lines.append(f"{cmap[a['category_id']]} {(x0+x1)/2/W:.6f} {(y0+y1)/2/H:.6f} {(x1-x0)/W:.6f} {(y1-y0)/H:.6f}")
        (YOLO_DIR / "labels" / s / (Path(fn).stem + ".txt")).write_text("\n".join(lines))
        n_lines += len(lines)
    counts[s] = n_lines
    print(f"{s}: images={ni} label_lines={n_lines} expected={na} -> {'OK' if n_lines == na else 'MISMATCH'}")
    if n_lines != na: raise SystemExit("STOP: label mismatch. Share this output.")
YAML = YOLO_DIR / "bepli.yaml"
with open(YAML, "w") as f:
    f.write(f"path: {YOLO_DIR}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n")
    for i, c in enumerate(cats): f.write(f"  {i}: {c['name']}\n")
print("YOLO DATASET OK |", f"{(time.time()-t0)/60:.1f} min elapsed")

# ---------- train ----------
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics"])
import torch, ultralytics
from ultralytics import YOLO
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if not torch.cuda.is_available(): raise SystemExit("STOP: no CUDA GPU. Enable GPU in Session options and rerun.")

NAME = "yolo11s_1280"; RUN = OUTD / NAME
last, best, done = RUN / "weights" / "last.pt", RUN / "weights" / "best.pt", RUN / "FINISHED"
if done.exists():
    print("Training already finished; skipping to evaluation.")
elif last.exists():
    print("Resuming from", last); YOLO(str(last)).train(resume=True)
else:
    YOLO("yolo11s.pt").train(data=str(YAML), imgsz=1280, epochs=60, time=3.0, batch=8, workers=4, patience=20,
                             cos_lr=True, close_mosaic=10, max_det=400, seed=0, plots=True, device=0,
                             project=str(OUTD), name=NAME, exist_ok=True)
done.write_text(f"finished {time.strftime('%Y-%m-%d %H:%M')}")

# ---------- evaluate best.pt on the official test split ----------
model = YOLO(str(best))
res = model.val(data=str(YAML), split="test", imgsz=1280, batch=8, max_det=400, conf=0.001, iou=0.6,
                plots=True, device=0, project=str(OUTD), name="test_eval", exist_ok=True)
print("\n=== OFFICIAL TEST SPLIT (744 images) ===")
print(f"precision={res.box.mp:.3f} recall={res.box.mr:.3f} mAP50={res.box.map50:.3f} mAP50-95={res.box.map:.3f}")
for i, n in res.names.items(): print(f"  {n:20s} mAP50-95={res.box.maps[i]:.3f}")

# ---------- zip results (no dataset) ----------
bk = Path("/kaggle/working/backup_stage"); shutil.rmtree(bk, ignore_errors=True); bk.mkdir()
shutil.copytree(RUN, bk / NAME)
if (OUTD / "test_eval").exists(): shutil.copytree(OUTD / "test_eval", bk / "test_eval")
shutil.copy(YAML, bk / "bepli.yaml")
z = shutil.make_archive("/kaggle/working/marine_run", "zip", str(bk)); shutil.rmtree(bk, ignore_errors=True)
print("\nBackup zip:", z, f"{os.path.getsize(z)/1e6:.0f} MB | total {(time.time()-t0)/60:.0f} min")
print("DONE. Download marine_run.zip from the Output panel (right side, /kaggle/working).")

In [ ]:
from pathlib import Path

run = Path("/kaggle/working/runs/yolo11s_1280")

for name in ["last.pt", "best.pt"]:
    p = run / "weights" / name
    print(name, "EXISTS" if p.exists() else "MISSING",
          f"({p.stat().st_size / 1024 / 1024:.1f} MB)" if p.exists() else "")

print("Results CSV:", (run / "results.csv").exists())
print("Training arguments:", (run / "args.yaml").exists())

In [ ]:
import shutil
from pathlib import Path

run = Path("/kaggle/working/runs/yolo11s_1280")
backup = Path("/kaggle/working/marine_training_checkpoint")

if backup.exists():
    shutil.rmtree(backup)

backup.mkdir(parents=True)

shutil.copytree(run / "weights", backup / "weights")
shutil.copy2(run / "results.csv", backup / "results.csv")
shutil.copy2(run / "args.yaml", backup / "args.yaml")

zip_path = shutil.make_archive(
    "/kaggle/working/marine_training_checkpoint",
    "zip",
    backup
)

print("BACKUP CREATED:", zip_path)
print("SIZE:", round(Path(zip_path).stat().st_size / (1024 * 1024), 2), "MB")


In [ ]:
from pathlib import Path

print("Kaggle input folders:")

for p in Path("/kaggle/input").iterdir():
    print("\n", p)
    if p.is_dir():
        for f in p.rglob("*"):
            if f.is_file():
                print("  ", f.relative_to(p))

In [ ]:
from pathlib import Path
import shutil

source = Path("/kaggle/input/datasets/vaanyasehrawat/marine-plastics")
destination = Path("/kaggle/working/runs/yolo11s_1280")

(destination / "weights").mkdir(parents=True, exist_ok=True)

# Restore model weights
for name in ["last.pt", "best.pt"]:
    shutil.copy2(
        source / "weights" / name,
        destination / "weights" / name
    )

# Restore training history and configuration
for name in ["args.yaml", "results.csv"]:
    shutil.copy2(source / name, destination / name)

print("CHECKPOINT RESTORATION COMPLETE")

for p in [
    destination / "weights/last.pt",
    destination / "weights/best.pt",
    destination / "args.yaml",
    destination / "results.csv"
]:
    print(p.name, "OK" if p.exists() else "MISSING")

In [ ]:
import torch
from pathlib import Path

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nDataset YAML exists:",
      Path("/tmp/yolo_bepli/bepli.yaml").exists())

print("Dataset directory exists:",
      Path("/tmp/yolo_bepli").exists())

print("Restored checkpoint exists:",
      Path("/kaggle/working/runs/yolo11s_1280/weights/last.pt").exists())

In [ ]:
import os, sys, json, time, shutil, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from PIL import Image
t0 = time.time()
Y = Path("/tmp/yolo_bepli"); OUTD = Path("/kaggle/working/runs")
BEST = OUTD / "yolo11s_1280" / "weights" / "best.pt"; YAML = Y / "bepli.yaml"
if not (BEST.exists() and YAML.exists()):
    raise SystemExit("STOP: best.pt or the YOLO dataset is missing in this session. Share this message.")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxslim", "onnxruntime"])
from ultralytics import YOLO
model = YOLO(str(BEST))

# ---------- ONNX export + check ----------
onnx_path = Path(model.export(format="onnx", imgsz=1280, opset=17, simplify=True, dynamic=False, half=False, device="cpu"))
print(f"ONNX saved: {onnx_path} ({onnx_path.stat().st_size/1e6:.1f} MB)")
import onnxruntime as ort
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
print("ONNX input:", sess.get_inputs()[0].name, sess.get_inputs()[0].shape, "| output:", sess.get_outputs()[0].name, sess.get_outputs()[0].shape)
def img_list(split): return [p for p in sorted((Y / "images" / split).iterdir()) if p.suffix.lower() in {".png", ".jpg", ".jpeg"}]
chk = img_list("test")[:20]
m_pt = YOLO(str(BEST)); m_ox = YOLO(str(onnx_path), task="detect")
npt = sum(len(m_pt.predict(str(p), imgsz=1280, conf=0.25, max_det=400, verbose=False, device="cpu")[0].boxes) for p in chk)
nox = sum(len(m_ox.predict(str(p), imgsz=1280, conf=0.25, max_det=400, verbose=False)[0].boxes) for p in chk)
print(f"ONNX CHECK on 20 test images: PyTorch dets={npt} ONNX dets={nox}")

# ---------- near-duplicate flags for the clean test ----------
TABLE = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def feat(p):
    im = Image.open(p).convert("L")
    a = np.asarray(im.resize((9, 8), Image.LANCZOS), dtype=np.int16)
    g = np.asarray(im.resize((24, 24), Image.LANCZOS)).astype(np.float32).ravel(); g -= g.mean(); g /= (np.linalg.norm(g) + 1e-6)
    return np.packbits((a[:, 1:] > a[:, :-1]).flatten()), g
def load(split):
    ps = img_list(split)
    with ThreadPoolExecutor(8) as ex: r = list(ex.map(feat, ps))
    return ps, np.stack([x[0] for x in r]), np.stack([x[1] for x in r])
print("Hashing train/val/test ...")
_, d1, g1 = load("train"); _, d2, g2 = load("val"); pte, dte, gte = load("test")
dref, gref = np.concatenate([d1, d2]), np.concatenate([g1, g2])
flag = set()
for k, r in enumerate(dte):
    d = TABLE[np.bitwise_xor(dref, r)].sum(axis=1); j = int(d.argmin())
    if d[j] <= 5 and float(gref[j] @ gte[k]) >= 0.90: flag.add(pte[k].stem)
print(f"Flagged near-duplicate test images ({len(flag)}):", sorted(flag))

# ---------- clean-test and class-agnostic evaluation ----------
if flag:
    ci, cl = Y / "images" / "test_clean", Y / "labels" / "test_clean"; ci.mkdir(exist_ok=True); cl.mkdir(exist_ok=True)
    for p in img_list("test"):
        if p.stem in flag: continue
        if not (ci / p.name).exists():
            try: os.link(p, ci / p.name)
            except OSError: shutil.copy(p, ci / p.name)
        shutil.copy(Y / "labels" / "test" / (p.stem + ".txt"), cl / (p.stem + ".txt"))
    (Y / "bepli_clean.yaml").write_text(YAML.read_text().replace("test: images/test\n", "test: images/test_clean\n"))
    r2 = model.val(data=str(Y / "bepli_clean.yaml"), split="test", imgsz=1280, batch=8, max_det=400, conf=0.001, iou=0.6,
                   device=0, project=str(OUTD), name="test_clean_eval", exist_ok=True, plots=False)
    print(f"\n=== CLEAN TEST ({744-len(flag)} images, {len(flag)} removed) === mAP50={r2.box.map50:.3f} mAP50-95={r2.box.map:.3f}")
else:
    print("\nNo flagged test images: clean test equals the official test.")
try:
    r3 = model.val(data=str(YAML), split="test", imgsz=1280, batch=8, max_det=400, conf=0.001, iou=0.6, single_cls=True,
                   device=0, project=str(OUTD), name="test_agnostic", exist_ok=True, plots=False)
    print(f"=== CLASS-AGNOSTIC TEST (any plastic, class ignored) === precision={r3.box.mp:.3f} recall={r3.box.mr:.3f} mAP50={r3.box.map50:.3f} mAP50-95={r3.box.map:.3f}")
except Exception as e:
    print("Class-agnostic eval skipped:", str(e)[:200])

# ---------- export bundle ----------
st = Path("/kaggle/working/export_stage"); shutil.rmtree(st, ignore_errors=True); st.mkdir()
shutil.copy(BEST, st / "yolo11s_1280_best.pt"); shutil.copy(onnx_path, st / "yolo11s_1280_best.onnx")
json.dump({"imgsz": 1280, "names": [model.names[i] for i in range(len(model.names))], "output_shape": [1, 4 + len(model.names), 33600],
           "flagged_test_images": sorted(flag)}, open(st / "model_info.json", "w"), indent=1)
z = shutil.make_archive("/kaggle/working/marine_export", "zip", str(st))
print(f"\nBundle: {z} ({os.path.getsize(z)/1e6:.0f} MB) | {(time.time()-t0)/60:.1f} min")
print("DONE-A")

In [ ]:
import json, time, shutil
from pathlib import Path
import numpy as np, torch, torchvision, onnxruntime as ort
from PIL import Image
from ultralytics import YOLO

Y = Path("/tmp/yolo_bepli")
PT = Path("/kaggle/working/runs/yolo11s_1280/weights/best.pt"); OX = PT.with_suffix(".onnx")
if not (PT.exists() and OX.exists() and Y.exists()):
    raise SystemExit("STOP: best.pt, best.onnx or the dataset is missing in this session. Share this message.")
S = 1280

def letterbox(im):
    w, h = im.size; r = min(S / w, S / h); nw, nh = round(w * r), round(h * r)
    left, top = int(round((S - nw) / 2 - 0.1)), int(round((S - nh) / 2 - 0.1))
    canvas = Image.new("RGB", (S, S), (114, 114, 114))
    canvas.paste(im.resize((nw, nh), Image.BILINEAR), (left, top))
    x = np.asarray(canvas, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
    return np.ascontiguousarray(x), r, left, top

def decode(out, conf_th=0.25, iou=0.7, max_det=400):
    p = torch.as_tensor(out)[0].T
    b, sc = p[:, :4], p[:, 4:]
    conf, cls = sc.max(1); m = conf > conf_th
    b, conf, cls = b[m], conf[m], cls[m]
    xyxy = torch.cat([b[:, :2] - b[:, 2:] / 2, b[:, :2] + b[:, 2:] / 2], 1)
    keep = torchvision.ops.batched_nms(xyxy, conf, cls, iou)[:max_det]
    return xyxy[keep], conf[keep], cls[keep]

net = YOLO(str(PT)).model.float().eval().cpu()
sess = ort.InferenceSession(str(OX), providers=["CPUExecutionProvider"])
iname = sess.get_inputs()[0].name
names = YOLO(str(PT)).names
files = sorted(p for p in (Y / "images" / "test").iterdir() if p.suffix.lower() == ".png")
pick = [files[0], files[len(files) // 2], files[-1]]

fx = Path("/kaggle/working/java_fixture"); shutil.rmtree(fx, ignore_errors=True); fx.mkdir()
fixture = {"imgsz": S, "conf": 0.25, "iou": 0.7, "max_det": 400, "letterbox": "resize keep ratio bilinear, pad 114, left=round((1280-nw)/2-0.1), top=round((1280-nh)/2-0.1)",
           "names": [names[i] for i in range(len(names))], "images": []}
worst_box = worst_score = 0.0
for p in pick:
    im = Image.open(p).convert("RGB"); x, r, left, top = letterbox(im)
    with torch.no_grad():
        o = net(torch.from_numpy(x))
    o_t = (o[0] if isinstance(o, (list, tuple)) else o).numpy()
    o_o = sess.run(None, {iname: x})[0]
    d_box = float(np.abs(o_t[:, :4] - o_o[:, :4]).max()); d_sc = float(np.abs(o_t[:, 4:] - o_o[:, 4:]).max())
    worst_box, worst_score = max(worst_box, d_box), max(worst_score, d_sc)
    bt, ct, kt = decode(o_t); bo, co, ko = decode(o_o)
    print(f"{p.name}: dets torch={len(bt)} onnx={len(bo)} | max|diff| boxes={d_box:.4f}px scores={d_sc:.6f}")
    shutil.copy(p, fx / p.name)
    fixture["images"].append({"file": p.name, "orig_w": im.size[0], "orig_h": im.size[1], "scale": r, "pad_left": left, "pad_top": top,
        "n_detections": int(len(bo)),
        "top10": [{"cls": int(k), "name": names[int(k)], "conf": round(float(c), 4), "box_letterbox_xyxy": [round(float(v), 2) for v in b]}
                  for b, c, k in list(zip(bo, co, ko))[:10]]})
json.dump(fixture, open(fx / "fixture.json", "w"), indent=1)
z = shutil.make_archive("/kaggle/working/java_fixture", "zip", str(fx))
print(f"\nWORST DIFF over 3 images: boxes={worst_box:.4f}px scores={worst_score:.6f}")
print("PARITY:", "PASS" if worst_box < 0.5 and worst_score < 0.01 else "FAIL - share this output")
print("Fixture zip:", z)
print("DONE-PARITY")

In [ ]:
import os, sys, json, time, shutil
from pathlib import Path
os.environ["WANDB_DISABLED"] = "true"
t0 = time.time()
Y = Path("/tmp/yolo_bepli"); YAML = Y / "bepli.yaml"; OUTD = Path("/kaggle/working/runs")
S_BEST = OUTD / "yolo11s_1280" / "weights" / "best.pt"
if not (YAML.exists() and S_BEST.exists()):
    raise SystemExit("STOP: dataset or YOLO11s weights missing in this session. Share this message.")
import torch
from ultralytics import YOLO
ng = torch.cuda.device_count(); print("GPUs:", ng, [torch.cuda.get_device_name(i) for i in range(ng)])
NAME = "yolo11m_1280"; RUN = OUTD / NAME
last, best, done = RUN / "weights" / "last.pt", RUN / "weights" / "best.pt", RUN / "FINISHED"

def fit(dev, batch, epochs):
    if last.exists():
        print("Resuming from", last); YOLO(str(last)).train(resume=True)
    else:
        YOLO("yolo11m.pt").train(data=str(YAML), imgsz=1280, epochs=epochs, time=3.0, batch=batch, workers=4, patience=15,
                                 cos_lr=True, close_mosaic=8, max_det=400, seed=0, plots=True, device=dev,
                                 project=str(OUTD), name=NAME, exist_ok=True)

if not done.exists():
    try:
        fit([0, 1], 8, 50) if ng >= 2 else fit(0, 4, 40)
    except Exception as e:
        print("Training failed:", str(e)[:400])
        if ng >= 2 and not best.exists():
            print("Falling back to a single GPU ...")
            shutil.rmtree(RUN, ignore_errors=True); torch.cuda.empty_cache(); fit(0, 4, 40)
        elif not best.exists():
            raise
    done.write_text(time.strftime("finished %Y-%m-%d %H:%M"))

def ev(w, split, tag):
    return YOLO(str(w)).val(data=str(YAML), split=split, imgsz=1280, batch=8, max_det=400, conf=0.001, iou=0.6,
                            device=0, project=str(OUTD), name=f"eval_{tag}_{split}", exist_ok=True, plots=(split == "test"))
R = {}
for tag, w in [("s", S_BEST), ("m", best)]:
    for split in ["val", "test"]:
        R[(tag, split)] = ev(w, split, tag)
print("\n=== COMPARISON (conf=0.001, iou=0.6, imgsz=1280) ===")
for tag, nm in [("s", "YOLO11s"), ("m", "YOLO11m")]:
    v, t = R[(tag, "val")].box, R[(tag, "test")].box
    print(f"{nm}: VAL mAP50={v.map50:.3f} mAP50-95={v.map:.3f} | TEST P={t.mp:.3f} R={t.mr:.3f} mAP50={t.map50:.3f} mAP50-95={t.map:.3f}")
rm = R[("m", "test")]
print("\nYOLO11m per-class TEST mAP50-95:")
for i, n in rm.names.items(): print(f"  {n:20s} {rm.box.maps[i]:.3f}")

sel = "m" if R[("m", "val")].box.map > R[("s", "val")].box.map else "s"
print(f"\nSELECTED MODEL (by val mAP50-95): YOLO11{sel}")
st = Path("/kaggle/working/export_b"); shutil.rmtree(st, ignore_errors=True); st.mkdir()
if sel == "m":
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxslim", "onnxruntime"])
    op = Path(YOLO(str(best)).export(format="onnx", imgsz=1280, opset=17, simplify=True, dynamic=False, half=False, device="cpu"))
    import onnxruntime as ort
    s_ = ort.InferenceSession(str(op), providers=["CPUExecutionProvider"])
    print("ONNX output shape:", s_.get_outputs()[0].shape)
    chk = [p for p in sorted((Y / "images" / "test").iterdir())][:20]
    a = YOLO(str(best)); b = YOLO(str(op), task="detect")
    na = sum(len(a.predict(str(p), imgsz=1280, conf=0.25, max_det=400, verbose=False, device="cpu")[0].boxes) for p in chk)
    nb = sum(len(b.predict(str(p), imgsz=1280, conf=0.25, max_det=400, verbose=False)[0].boxes) for p in chk)
    print(f"ONNX CHECK (20 test images): PyTorch dets={na} ONNX dets={nb}")
    shutil.copy(best, st / "yolo11m_1280_best.pt"); shutil.copy(op, st / "yolo11m_1280_best.onnx")
shutil.copytree(RUN, st / NAME, ignore=shutil.ignore_patterns("last.pt", "epoch*.pt"))
for d in OUTD.glob("eval_*"): shutil.copytree(d, st / d.name, dirs_exist_ok=True)
z = shutil.make_archive("/kaggle/working/marine_run2", "zip", str(st))
print(f"\nBundle: {z} ({os.path.getsize(z)/1e6:.0f} MB) | total {(time.time()-t0)/60:.0f} min")
print("DONE-B. Download marine_run2.zip from the Output panel (/kaggle/working).")

In [ ]:
import json, time, shutil
from pathlib import Path
import numpy as np, torch, torchvision, onnxruntime as ort
from PIL import Image
from ultralytics import YOLO

t0 = time.time()
Y = Path("/tmp/yolo_bepli")
PT = Path("/kaggle/working/runs/yolo11m_1280/weights/best.pt"); OX = PT.with_suffix(".onnx")
if not (PT.exists() and OX.exists() and Y.exists()):
    raise SystemExit("STOP: YOLO11m best.pt/best.onnx or the dataset is missing in this session. Share this message.")
S = 1280
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", dev)

def letterbox(im):
    w, h = im.size; r = min(S / w, S / h); nw, nh = round(w * r), round(h * r)
    left, top = int(round((S - nw) / 2 - 0.1)), int(round((S - nh) / 2 - 0.1))
    canvas = Image.new("RGB", (S, S), (114, 114, 114))
    canvas.paste(im.resize((nw, nh), Image.BILINEAR), (left, top))
    x = np.asarray(canvas, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
    return np.ascontiguousarray(x), r, left, top

def decode(out, conf_th=0.25, iou=0.7, max_det=400):
    p = torch.as_tensor(out)[0].T
    b, sc = p[:, :4], p[:, 4:]
    conf, cls = sc.max(1); m = conf > conf_th
    b, conf, cls = b[m], conf[m], cls[m]
    xyxy = torch.cat([b[:, :2] - b[:, 2:] / 2, b[:, :2] + b[:, 2:] / 2], 1)
    keep = torchvision.ops.batched_nms(xyxy, conf, cls, iou)[:max_det]
    return xyxy[keep], conf[keep], cls[keep]

names = YOLO(str(PT)).names
net_cpu = YOLO(str(PT)).model.float().eval().cpu()
net_gpu = YOLO(str(PT)).model.float().eval().to(dev)
sess = ort.InferenceSession(str(OX), providers=["CPUExecutionProvider"])
iname = sess.get_inputs()[0].name
def head(o): return (o[0] if isinstance(o, (list, tuple)) else o)

# ---------- parity + Java fixture ----------
files = sorted(p for p in (Y / "images" / "test").iterdir() if p.suffix.lower() == ".png")
pick = [files[0], files[len(files) // 2], files[-1]]
fx = Path("/kaggle/working/java_fixture_m"); shutil.rmtree(fx, ignore_errors=True); fx.mkdir()
fixture = {"imgsz": S, "conf": 0.25, "iou": 0.7, "max_det": 400, "model": "yolo11m_1280",
           "letterbox": "resize keep ratio bilinear, pad 114, left=round((1280-nw)/2-0.1), top=round((1280-nh)/2-0.1)",
           "names": [names[i] for i in range(len(names))], "images": []}
wb = ws = 0.0
for p in pick:
    im = Image.open(p).convert("RGB"); x, r, left, top = letterbox(im)
    with torch.no_grad(): o_t = head(net_cpu(torch.from_numpy(x))).numpy()
    o_o = sess.run(None, {iname: x})[0]
    d_b = float(np.abs(o_t[:, :4] - o_o[:, :4]).max()); d_s = float(np.abs(o_t[:, 4:] - o_o[:, 4:]).max())
    wb, ws = max(wb, d_b), max(ws, d_s)
    bt, ct, kt = decode(o_t); bo, co, ko = decode(o_o)
    print(f"{p.name}: dets torch={len(bt)} onnx={len(bo)} | max|diff| boxes={d_b:.4f}px scores={d_s:.6f}")
    shutil.copy(p, fx / p.name)
    fixture["images"].append({"file": p.name, "orig_w": im.size[0], "orig_h": im.size[1], "scale": r, "pad_left": left, "pad_top": top,
        "n_detections": int(len(bo)),
        "top10": [{"cls": int(k), "name": names[int(k)], "conf": round(float(c), 4), "box_letterbox_xyxy": [round(float(v), 2) for v in b]}
                  for b, c, k in list(zip(bo, co, ko))[:10]]})
json.dump(fixture, open(fx / "fixture.json", "w"), indent=1)
zf = shutil.make_archive("/kaggle/working/java_fixture_m", "zip", str(fx))
ok = wb < 0.5 and ws < 0.01
print(f"WORST DIFF: boxes={wb:.4f}px scores={ws:.6f} | PARITY: {'PASS' if ok else 'FAIL'} | fixture zip: {zf}")
if not ok: raise SystemExit("STOP: YOLO11m ONNX parity failed. Share this output.")

# ---------- detection dump (same letterbox + decode as Java, conf >= 0.10) ----------
def read_gt(split, stem, w, h):
    f = Y / "labels" / split / (stem + ".txt"); out = []
    if f.exists():
        for line in f.read_text().splitlines():
            q = line.split()
            if len(q) < 5: continue
            c = int(q[0]); cx, cy, bw, bh = [float(v) for v in q[1:5]]
            out.append([round((cx - bw / 2) * w, 1), round((cy - bh / 2) * h, 1), round((cx + bw / 2) * w, 1), round((cy + bh / 2) * h, 1), c])
    return out

DATA = {}
for split in ["val", "test"]:
    fl = sorted(p for p in (Y / "images" / split).iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"})
    if len(fl) != 744: raise SystemExit(f"STOP: expected 744 {split} images, found {len(fl)}")
    recs = []
    for n, p in enumerate(fl, 1):
        im = Image.open(p).convert("RGB"); w, h = im.size
        x, r, left, top = letterbox(im)
        with torch.no_grad(): o = head(net_gpu(torch.from_numpy(x).to(dev))).float().cpu().numpy()
        b, c, k = decode(o, 0.10)
        b = b.numpy().copy()
        b[:, [0, 2]] = np.clip((b[:, [0, 2]] - left) / r, 0, w); b[:, [1, 3]] = np.clip((b[:, [1, 3]] - top) / r, 0, h)
        recs.append({"file": p.name, "w": w, "h": h,
                     "dets": [[round(float(v), 1) for v in bb] + [round(float(cc), 4), int(kk)] for bb, cc, kk in zip(b, c.numpy(), k.numpy())],
                     "gt": read_gt(split, p.stem, w, h)})
        if n % 250 == 0: print(f"  {split} {n}/{len(fl)} ({time.time()-t0:.0f}s)")
    DATA[split] = recs
json.dump(DATA, open("/kaggle/working/route_data.json", "w"))
for s in DATA:
    print(f"{s}: images={len(DATA[s])} mean detections/img={np.mean([len(r['dets']) for r in DATA[s]]):.1f} mean GT/img={np.mean([len(r['gt']) for r in DATA[s]]):.1f}")
print(f"route_data.json saved | {(time.time()-t0)/60:.1f} min")
print("DONE-C")

In [ ]:
import json, math, time, gzip, shutil, subprocess, sys
from pathlib import Path
import numpy as np, torch, torch.nn as nn, torch.nn.functional as Fn
import onnxruntime as ort
from scipy.stats import spearmanr

t0 = time.time()
DP = Path("/kaggle/working/route_data.json")
if not DP.exists(): raise SystemExit("STOP: route_data.json missing. Run Cell C first.")
DATA = json.load(open(DP))
dev = "cuda" if torch.cuda.is_available() else "cpu"
R, C = 6, 8; T = R * C; NC = 13; NF = NC + 2; CMIN = 0.10
NAMES = ["pet_bottle", "other_bottle", "plastic_bag", "box_shaped_case", "other_container", "rope", "other_string",
         "fishing_net", "buoy", "other_fishing_gear", "styrene_foam", "others", "fragment"]
WEIGHT = {"fishing_net": 3.0, "other_fishing_gear": 3.0, "buoy": 2.5, "pet_bottle": 2.0, "other_bottle": 2.0, "plastic_bag": 2.0,
          "box_shaped_case": 2.0, "other_container": 2.0, "rope": 2.0, "other_string": 1.5, "styrene_foam": 1.5, "others": 1.0, "fragment": 0.5}
wv = np.array([WEIGHT[n] for n in NAMES], np.float32)

def tile_of(cx, cy, w, h):
    c = min(C - 1, max(0, int(math.floor(cx / w * C)))); r = min(R - 1, max(0, int(math.floor(cy / h * R))))
    return r * C + c

def feats(rec):
    f = np.zeros((T, NF), np.float32)
    for x1, y1, x2, y2, cf, k in rec["dets"]:
        if cf < CMIN: continue
        t = tile_of((x1 + x2) / 2, (y1 + y2) / 2, rec["w"], rec["h"]); k = int(k)
        f[t, k] += cf; f[t, NC] += 1; f[t, NC + 1] = max(f[t, NC + 1], cf)
    return np.log1p(f)

def target(rec):
    y = np.zeros(T, np.float32)
    for x1, y1, x2, y2, k in rec["gt"]: y[tile_of((x1 + x2) / 2, (y1 + y2) / 2, rec["w"], rec["h"])] += wv[int(k)]
    return y

def base_score(rec):
    s = np.zeros(T, np.float32)
    for x1, y1, x2, y2, cf, k in rec["dets"]:
        if cf >= 0.25: s[tile_of((x1 + x2) / 2, (y1 + y2) / 2, rec["w"], rec["h"])] += wv[int(k)] * cf
    return s

def build(split):
    r = DATA[split]
    return np.stack([feats(q) for q in r]), np.stack([target(q) for q in r]), np.stack([base_score(q) for q in r])
Xv, Yv, Bv = build("val"); Xt, Yt, Bt = build("test")
rng0 = np.random.default_rng(0); perm = rng0.permutation(len(Xv)); tr_i, ho_i = perm[:600], perm[600:]
tt = lambda a: torch.tensor(a, dtype=torch.float32, device=dev)
Xtr, Ytr, Xho, Yho = tt(Xv[tr_i]), tt(np.log1p(Yv[tr_i])), tt(Xv[ho_i]), tt(np.log1p(Yv[ho_i]))
print(f"val images {len(Xv)} (train {len(tr_i)} / holdout {len(ho_i)}), test images {len(Xt)}, tiles {T}, features {NF}")

class Block(nn.Module):
    def __init__(s, d, h, ff, p):
        super().__init__()
        s.h = h; s.qkv = nn.Linear(d, 3 * d); s.o = nn.Linear(d, d)
        s.n1 = nn.LayerNorm(d); s.n2 = nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d)); s.dp = nn.Dropout(p)
    def forward(s, x):
        B, Tn, D = x.shape; dh = D // s.h
        q, k, v = s.qkv(s.n1(x)).chunk(3, dim=-1)
        q = q.reshape(B, Tn, s.h, dh).transpose(1, 2); k = k.reshape(B, Tn, s.h, dh).transpose(1, 2); v = v.reshape(B, Tn, s.h, dh).transpose(1, 2)
        a = torch.softmax(q @ k.transpose(-1, -2) / math.sqrt(dh), dim=-1)
        y = (a @ v).transpose(1, 2).reshape(B, Tn, D)
        x = x + s.dp(s.o(y))
        return x + s.dp(s.ff(s.n2(x)))

class RouteTransformer(nn.Module):
    def __init__(s, d=64, h=4, L=2, ff=128, p=0.1):
        super().__init__()
        s.inp = nn.Linear(NF, d); s.pos = nn.Parameter(torch.zeros(1, T, d)); nn.init.normal_(s.pos, std=0.02)
        s.blocks = nn.ModuleList([Block(d, h, ff, p) for _ in range(L)]); s.norm = nn.LayerNorm(d); s.out = nn.Linear(d, 1)
    def forward(s, x):
        z = s.inp(x) + s.pos
        for b in s.blocks: z = b(z)
        return s.out(s.norm(z)).squeeze(-1)

class TileMLP(nn.Module):
    def __init__(s, d=64, p=0.1):
        super().__init__()
        s.inp = nn.Linear(NF, d); s.pos = nn.Parameter(torch.zeros(1, T, d)); nn.init.normal_(s.pos, std=0.02)
        s.net = nn.Sequential(nn.GELU(), nn.Dropout(p), nn.Linear(d, 128), nn.GELU(), nn.Dropout(p), nn.Linear(128, d), nn.GELU()); s.out = nn.Linear(d, 1)
    def forward(s, x): return s.out(s.net(s.inp(x) + s.pos)).squeeze(-1)

def flip(x):
    sh = x.shape
    return x.reshape(sh[0], R, C, *sh[2:]).flip(2).reshape(sh)

def fit(kind, seed, EP=200):
    torch.manual_seed(seed); np.random.seed(seed)
    m = (RouteTransformer() if kind == "transformer" else TileMLP()).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=2e-3, weight_decay=1e-2)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EP)
    best, state, bad = 1e9, None, 0
    for ep in range(EP):
        m.train(); pm = torch.randperm(len(Xtr), device=dev)
        for i in range(0, len(pm), 32):
            ix = pm[i:i + 32]; x, y = Xtr[ix], Ytr[ix]
            if torch.rand(1).item() < 0.5: x, y = flip(x), flip(y)
            loss = Fn.mse_loss(m(x), y); opt.zero_grad(); loss.backward(); opt.step()
        sch.step(); m.eval()
        with torch.no_grad(): vl = Fn.mse_loss(m(Xho), Yho).item()
        if vl < best - 1e-5: best, state, bad = vl, {k: v.detach().clone() for k, v in m.state_dict().items()}, 0
        else: bad += 1
        if bad >= 25: break
    m.load_state_dict(state); m.eval()
    return m, best

def predict(m, X):
    with torch.no_grad(): return m(tt(X)).cpu().numpy()

def metrics(P, Yg, seed=0):
    rng = np.random.default_rng(seed); sp, ov = [], []
    for p, y in zip(P, Yg):
        if (y > 0).sum() < 5: continue
        pj = p + 1e-9 * rng.random(T)
        s = spearmanr(pj, y)[0]; sp.append(0.0 if not np.isfinite(s) else float(s))
        gt5 = set(np.argsort(-(y + 1e-6 * rng.random(T)))[:5].tolist()); pr5 = set(np.argsort(-pj)[:5].tolist())
        ov.append(len(gt5 & pr5) / 5.0)
    return float(np.mean(sp)), float(np.mean(ov)), len(sp)

out = {"random": metrics(np.random.default_rng(5).random(Yt.shape), Yt), "detector_only_conf0.25": metrics(Bt, Yt)}
models, summary = {}, {}
for kind in ["tile_mlp", "transformer"]:
    runs = []
    for seed in [0, 1, 2]:
        m, hl = fit(kind, seed); sp, ov, n = metrics(predict(m, Xt), Yt)
        runs.append((sp, ov, hl)); models[(kind, seed)] = (m, hl)
        print(f"  {kind} seed {seed}: holdout MSE {hl:.4f} | test Spearman {sp:.3f} top-5 overlap {ov:.3f}")
    a = np.array(runs)
    summary[kind] = {"spearman_mean": float(a[:, 0].mean()), "spearman_std": float(a[:, 0].std()), "top5_mean": float(a[:, 1].mean()),
                     "top5_std": float(a[:, 1].std()), "params": int(sum(p.numel() for p in models[(kind, 0)][0].parameters()))}
n_eval = out["random"][2]
print(f"\n=== TILE PRIORITY RANKING ON TEST ({n_eval} of {len(Xt)} images have >=5 ground-truth tiles) ===")
print(f"{'method':34s} {'Spearman':>16s} {'top-5 overlap':>16s}")
print(f"{'random ranking':34s} {out['random'][0]:16.3f} {out['random'][1]:16.3f}")
print(f"{'detector-only planner score':34s} {out['detector_only_conf0.25'][0]:16.3f} {out['detector_only_conf0.25'][1]:16.3f}")
for kind, nm in [("tile_mlp", "tile MLP (no tile context)"), ("transformer", "route-context transformer")]:
    s = summary[kind]
    print(f"{nm + ' [' + str(s['params']) + ' params]':34s} {s['spearman_mean']:9.3f}+-{s['spearman_std']:.3f} {s['top5_mean']:9.3f}+-{s['top5_std']:.3f}")

# ---------- export best transformer (seed chosen by holdout loss, not by test) ----------
bs = min([0, 1, 2], key=lambda s: models[("transformer", s)][1]); bm = models[("transformer", bs)][0].cpu().eval()
OXP = Path("/kaggle/working/route_model.onnx")
try:
    torch.onnx.export(bm, torch.zeros(1, T, NF), str(OXP), input_names=["tiles"], output_names=["priority"], opset_version=17, dynamo=False)
except Exception as e:
    print("TorchScript export failed, trying dynamo exporter:", str(e)[:200])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnxscript"])
    torch.onnx.export(bm, torch.zeros(1, T, NF), str(OXP), input_names=["tiles"], output_names=["priority"], opset_version=17)
sess = ort.InferenceSession(str(OXP), providers=["CPUExecutionProvider"])
sel = [0, len(DATA["test"]) // 2, len(DATA["test"]) - 1]; fx, worst = [], 0.0
for i in sel:
    x = Xt[i:i + 1].astype(np.float32)
    with torch.no_grad(): o_t = bm(torch.from_numpy(x)).numpy()
    o_o = sess.run(None, {"tiles": x})[0]; worst = max(worst, float(np.abs(o_t - o_o).max()))
    r = DATA["test"][i]
    fx.append({"file": r["file"], "w": r["w"], "h": r["h"], "dets": r["dets"], "tiles": np.round(x[0], 7).tolist(),
               "priority": np.round(o_t[0], 7).tolist(), "gt_priority": Yt[i].tolist()})
print(f"\nONNX CHECK: best seed {bs}, max |torch - onnx| = {worst:.2e} | {'PASS' if worst < 1e-4 else 'FAIL'}")
info = {"rows": R, "cols": C, "n_features": NF, "conf_min": CMIN, "names": NAMES, "weights": WEIGHT, "input": "tiles [1,48,15]",
        "features": "per tile: log1p of [sum of detection confidences for each of 13 classes, detection count, max confidence]; detection centre decides the tile; detections with conf>=0.10 from YOLO11m (letterbox 1280, NMS iou 0.7)",
        "output": "priority [1,48], log1p scale of weighted ground-truth priority", "seed": bs, "metrics": {"random": out["random"], "detector_only": out["detector_only_conf0.25"], **summary}}
st = Path("/kaggle/working/route_stage"); shutil.rmtree(st, ignore_errors=True); st.mkdir()
shutil.copy(OXP, st / "route_model.onnx")
json.dump(info, open(st / "route_model_info.json", "w"), indent=1); json.dump(fx, open(st / "route_fixture.json", "w"))
with open(DP, "rb") as fi, gzip.open(st / "route_data.json.gz", "wb") as fo: shutil.copyfileobj(fi, fo)
z = shutil.make_archive("/kaggle/working/route_bundle", "zip", str(st))
print(f"Bundle: {z} | {(time.time()-t0)/60:.1f} min")
print("DONE-D")

In [ ]:
import os, sys, json, time, shutil, subprocess, hashlib, zipfile
from pathlib import Path
os.environ["WANDB_DISABLED"] = "true"
t0 = time.time()

WORK = Path("/kaggle/working"); WORK.mkdir(parents=True, exist_ok=True)
STATE = WORK / "env_state.json"
KNOWN = {"s": "79e633ff7e59195a82c01ecb898ad0800e42b6bf4c2a8bab8e29dcf70d6305bf",   # yolo11s_1280_best.pt (Mac SHA-256)
         "m": "d48ddb77496542cfa8413c652e74289fde48d1de5ab640522252845a5ad70031"}   # yolo11m_1280_best.pt (Mac SHA-256)
NAMES = ["pet_bottle", "other_bottle", "plastic_bag", "box_shaped_case", "other_container", "rope", "other_string",
         "fishing_net", "buoy", "other_fishing_gear", "styrene_foam", "others", "fragment"]
EXPECTED = {"train": (2234, 71889), "val": (744, 24643), "test": (744, 22040)}
URL = "https://www.seanoe.org/data/00858/96963/data/106157.tar.gz"
BASE = Path("/tmp/bepli"); RAW = BASE / "raw"; ARCH = BASE / "bepli.tar.gz"; OKFLAG = BASE / "EXTRACT_OK"
YD = Path("/tmp/yolo_bepli"); IMG_EXT = {".png", ".jpg", ".jpeg"}

# ---------- 1. GPU ----------
import torch
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip().replace("\n", " | ")
print("GPU:", gpu or "none")
if not torch.cuda.is_available():
    raise SystemExit("STOP: no CUDA GPU. Session options -> Accelerator -> GPU T4 x2 (or P100), then rerun.")
print("torch", torch.__version__, "| device 0:", torch.cuda.get_device_name(0))

# ---------- 2. ultralytics ----------
try:
    import ultralytics
except Exception:
    print("Installing ultralytics ...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.212"])
    except Exception:
        raise SystemExit("STOP: pip install failed. Session options -> Internet -> On, then rerun.")
    import ultralytics
from ultralytics import YOLO
print("ultralytics", ultralytics.__version__, "(detectors were trained with 8.4.212)")

# ---------- helpers ----------
def find_files(root, exts, maxdepth=7):
    root = Path(root); out = []
    if not root.exists(): return out
    d0 = len(root.parts)
    for dp, dn, fn in os.walk(root):
        if len(Path(dp).parts) - d0 >= maxdepth: dn[:] = []
        for f in fn:
            if f.lower().endswith(exts): out.append(Path(dp) / f)
    return sorted(out)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

def extract_zips():
    out_dir = WORK / "ckpt_extract"
    for z in find_files("/kaggle/input", (".zip",)):
        try:
            with zipfile.ZipFile(z) as zf:
                for m in zf.namelist():
                    if m.lower().endswith(".pt"): zf.extract(m, out_dir / z.stem)
        except Exception as e:
            print("  zip skipped:", z.name, str(e)[:80])
    return find_files(out_dir, (".pt",))

def info(p):
    try:
        m = YOLO(str(p))
        return sum(x.numel() for x in m.model.parameters()), [m.names[i] for i in range(len(m.names))]
    except Exception:
        return None

# ---------- 3. checkpoints ----------
pts = find_files("/kaggle/input", (".pt",))
print(f"Found {len(pts)} .pt files under /kaggle/input")
H = {p: sha256(p) for p in pts}
match = lambda H: {k: [p for p, h in H.items() if h == KNOWN[k]] for k in KNOWN}
M = match(H)
if not all(M.values()):
    extra = extract_zips()
    print(f"Extracted/found {len(extra)} .pt files from zip archives")
    H.update({p: sha256(p) for p in extra}); M = match(H)
CH, MATCH = {}, {}
for k in ("s", "m"):
    if M[k]: CH[k], MATCH[k] = M[k][0], True
for k in ("s", "m"):
    if k in CH: continue
    best = None
    for p in H:
        if p in CH.values(): continue
        r = info(p)
        if r is None or r[1] != NAMES: continue
        if ("s" if r[0] < 14e6 else "m") != k: continue
        score = ("best" in p.name.lower(), "last" not in p.name.lower())
        if best is None or score > best[0]: best = (score, p)
    if best: CH[k], MATCH[k] = best[1], False
if len(CH) < 2:
    print("All .pt files seen:"); [print("  ", p, h[:12]) for p, h in H.items()]
    raise SystemExit("STOP: YOLO11s and/or YOLO11m checkpoint not found. Attach 'Marine-plastics' (Add Input) and share this listing.")
for k, nm in (("s", "YOLO11s"), ("m", "YOLO11m")):
    r = info(CH[k])
    if r is None or r[1] != NAMES: raise SystemExit(f"STOP: {CH[k]} is not a 13-class BePLi checkpoint.")
    print(f"{nm}: {CH[k]} | params {r[0]:,} | SHA-256 {'MATCHES the Mac copy' if MATCH[k] else 'DIFFERS from the Mac copy (detector comparison may not describe the Java model)'}")

# ---------- 4. BePLi YOLO dataset ----------
def count_ok():
    if not (YD / "bepli.yaml").exists(): return False
    for s, (ni, na) in EXPECTED.items():
        di, dl = YD / "images" / s, YD / "labels" / s
        if not (di.is_dir() and dl.is_dir()): return False
        if sum(1 for p in di.iterdir() if p.suffix.lower() in IMG_EXT) != ni: return False
        lfs = list(dl.glob("*.txt"))
        if len(lfs) != ni: return False
        if sum(sum(1 for l in f.read_text().splitlines() if l.strip()) for f in lfs) != na: return False
    return True

def find_root(base):
    for dp, dn, fn in os.walk(base):
        if os.path.basename(dp) == "plastic_coco" and os.path.isdir(os.path.join(dp, "annotation")) and os.path.isdir(os.path.join(dp, "images")):
            return Path(dp)
    return None

def convert(ROOT):
    shutil.rmtree(YD, ignore_errors=True)
    cmap = None
    for s, (ni, na) in EXPECTED.items():
        js = [q for q in sorted((ROOT / "annotation").glob("*.json")) if s in q.name.lower()]
        if not js: raise SystemExit(f"STOP: no annotation JSON for {s} in {ROOT / 'annotation'}")
        d = json.load(open(js[0]))
        if len(d["images"]) != ni or len(d["annotations"]) != na:
            raise SystemExit(f"STOP: {s} has {len(d['images'])} images / {len(d['annotations'])} annotations, expected {ni} / {na}")
        if cmap is None:
            cats = sorted(d["categories"], key=lambda c: c["id"])
            if [c["name"] for c in cats] != NAMES: raise SystemExit("STOP: class names or order differ from the trained models.")
            cmap = {c["id"]: i for i, c in enumerate(cats)}
        (YD / "images" / s).mkdir(parents=True, exist_ok=True); (YD / "labels" / s).mkdir(parents=True, exist_ok=True)
        by = {}
        for a in d["annotations"]: by.setdefault(a["image_id"], []).append(a)
        n_lines = 0
        for im in d["images"]:
            fn = Path(im["file_name"]).name
            src = ROOT / "images" / s / fn
            if not src.exists(): src = ROOT / "images" / s / im["file_name"]
            dst = YD / "images" / s / fn
            if not dst.exists():
                try: os.link(src, dst)
                except OSError: shutil.copy(src, dst)
            W, Hh = im["width"], im["height"]; lines = []
            for a in by.get(im["id"], []):
                x, y, w, h = a["bbox"]
                x0, y0, x1, y1 = max(0, x), max(0, y), min(W, x + w), min(Hh, y + h)
                if x1 <= x0 or y1 <= y0: continue
                lines.append(f"{cmap[a['category_id']]} {(x0+x1)/2/W:.6f} {(y0+y1)/2/Hh:.6f} {(x1-x0)/W:.6f} {(y1-y0)/Hh:.6f}")
            (YD / "labels" / s / (Path(fn).stem + ".txt")).write_text("\n".join(lines))
            n_lines += len(lines)
        if n_lines != na: raise SystemExit(f"STOP: {s} label lines {n_lines} != {na}")
        print(f"  converted {s}: images={ni} label_lines={n_lines}")
    (YD / "bepli.yaml").write_text(f"path: {YD}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES)))

if count_ok():
    print("YOLO dataset already present and verified in /tmp/yolo_bepli.")
else:
    ROOT = find_root(RAW) if OKFLAG.exists() else None
    if ROOT: print("Using extracted BePLi at", ROOT)
    if ROOT is None:
        ROOT = find_root("/kaggle/input")
        if ROOT: print("Using BePLi attached under /kaggle/input:", ROOT)
    if ROOT is None:
        BASE.mkdir(parents=True, exist_ok=True); RAW.mkdir(parents=True, exist_ok=True)
        free = shutil.disk_usage("/tmp").free / 1e9
        print(f"/tmp free: {free:.0f} GB")
        if free < 25: raise SystemExit("STOP: less than 25 GB free in /tmp.")
        if not ARCH.exists() or ARCH.stat().st_size < 7e9:
            print("Downloading BePLi v2 (about 9 GB, roughly 10-15 min) ...")
            rc = subprocess.call(["wget", "-q", "-c", "--tries=5", "--timeout=60", "-O", str(ARCH), URL])
            if rc != 0: raise SystemExit(f"STOP: download failed (wget exit {rc}). Session options -> Internet -> On, then rerun.")
        print(f"Archive {ARCH.stat().st_size/1e9:.2f} GB | testing integrity ...")
        if subprocess.call(["gzip", "-t", str(ARCH)]) != 0:
            raise SystemExit("STOP: archive corrupt. Delete /tmp/bepli/bepli.tar.gz and rerun.")
        print("Extracting ...")
        if subprocess.call(["tar", "-xzf", str(ARCH), "-C", str(RAW)]) != 0: raise SystemExit("STOP: extraction failed.")
        ROOT = find_root(RAW)
        if ROOT is None: raise SystemExit("STOP: plastic_coco folder not found after extraction.")
        OKFLAG.write_text("ok"); os.remove(ARCH)
    print("Converting COCO -> YOLO (official splits unchanged) ...")
    convert(ROOT)
if not count_ok(): raise SystemExit("STOP: YOLO dataset failed verification after conversion.")
sets = {s: {p.name for p in (YD / "images" / s).iterdir()} for s in EXPECTED}
ov = len(sets["train"] & sets["val"]) + len(sets["train"] & sets["test"]) + len(sets["val"] & sets["test"])
for s, (ni, na) in EXPECTED.items(): print(f"{s}: images={ni} label_lines={na} -> OK")
print("filename overlap across splits:", ov)
if ov: raise SystemExit("STOP: split overlap.")

json.dump({"S_PT": str(CH["s"]), "M_PT": str(CH["m"]), "S_SHA": H[CH["s"]], "M_SHA": H[CH["m"]], "MATCH": MATCH,
           "YAML": str(YD / "bepli.yaml"), "YOLO_DIR": str(YD), "GPU": gpu}, open(STATE, "w"), indent=1)
print(f"\nSETUP OK | {(time.time()-t0)/60:.1f} min | state: {STATE}")

In [ ]:
import os, sys, json, time, shutil, subprocess, hashlib, zipfile
from pathlib import Path
os.environ["WANDB_DISABLED"] = "true"
t0 = time.time()

ALLOW_UNMATCHED = False   # keep False: the comparison must describe the exact models used in Java
WORK = Path("/kaggle/working"); WORK.mkdir(parents=True, exist_ok=True)
STATE = WORK / "env_state.json"
KNOWN = {"s": "79e633ff7e59195a82c01ecb898ad0800e42b6bf4c2a8bab8e29dcf70d6305bf",   # yolo11s_1280_best.pt (Mac SHA-256)
         "m": "d48ddb77496542cfa8413c652e74289fde48d1de5ab640522252845a5ad70031"}   # yolo11m_1280_best.pt (Mac SHA-256)
NAMES = ["pet_bottle", "other_bottle", "plastic_bag", "box_shaped_case", "other_container", "rope", "other_string",
         "fishing_net", "buoy", "other_fishing_gear", "styrene_foam", "others", "fragment"]
EXPECTED = {"train": (2234, 71889), "val": (744, 24643), "test": (744, 22040)}
URL = "https://www.seanoe.org/data/00858/96963/data/106157.tar.gz"
BASE = Path("/tmp/bepli"); RAW = BASE / "raw"; ARCH = BASE / "bepli.tar.gz"; OKFLAG = BASE / "EXTRACT_OK"
YD = Path("/tmp/yolo_bepli"); IMG_EXT = {".png", ".jpg", ".jpeg"}

# ---------- 1. GPU ----------
import torch
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip().replace("\n", " | ")
print("GPU:", gpu or "none")
if not torch.cuda.is_available():
    raise SystemExit("STOP: no CUDA GPU. Session options -> Accelerator -> GPU T4 x2 (or P100), then rerun.")
print("torch", torch.__version__, "| device 0:", torch.cuda.get_device_name(0))

# ---------- 2. ultralytics ----------
try:
    import ultralytics
except Exception:
    print("Installing ultralytics ...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.212"])
    except Exception:
        raise SystemExit("STOP: pip install failed. Session options -> Internet -> On, then rerun.")
    import ultralytics
from ultralytics import YOLO
print("ultralytics", ultralytics.__version__, "(detectors were trained with 8.4.212)")

# ---------- helpers ----------
def find_files(root, exts, maxdepth=8):
    root = Path(root); out = []
    if not root.exists(): return out
    d0 = len(root.parts)
    for dp, dn, fn in os.walk(root):
        if len(Path(dp).parts) - d0 >= maxdepth: dn[:] = []
        for f in fn:
            if f.lower().endswith(exts): out.append(Path(dp) / f)
    return sorted(out)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

def extract_zips():
    out_dir = WORK / "ckpt_extract"
    for z in find_files("/kaggle/input", (".zip",)):
        try:
            with zipfile.ZipFile(z) as zf:
                for m in zf.namelist():
                    if m.lower().endswith(".pt"): zf.extract(m, out_dir / z.stem)
        except Exception as e:
            print("  zip skipped:", z.name, str(e)[:80])
    return find_files(out_dir, (".pt",))

def describe(p, h):
    print(f"  {p}\n    size {p.stat().st_size/1e6:.1f} MB | SHA-256 {h[:16]}...")
    try:
        ck = torch.load(p, map_location="cpu", weights_only=False)
        ta = ck.get("train_args", {}) if isinstance(ck, dict) else {}
        print(f"    train_args: model={ta.get('model')} data={ta.get('data')} imgsz={ta.get('imgsz')} epochs={ta.get('epochs')}")
        m = ck.get("model") if isinstance(ck, dict) else None
        if m is not None:
            nm = getattr(m, "names", None); n = sum(x.numel() for x in m.parameters())
            nl = [nm[i] for i in range(len(nm))] if nm else None
            print(f"    parameters {n:,} | classes {len(nl) if nl else '?'} | BePLi class order matches: {nl == NAMES}")
    except Exception as e:
        print("    could not inspect:", str(e)[:120])

def info(p):
    try:
        m = YOLO(str(p)); return sum(x.numel() for x in m.model.parameters()), [m.names[i] for i in range(len(m.names))]
    except Exception:
        return None

# ---------- 3. checkpoints (identified by SHA-256) ----------
pts = find_files("/kaggle/input", (".pt",))
print(f"Found {len(pts)} .pt files under /kaggle/input")
H = {p: sha256(p) for p in pts}
match = lambda H: {k: [p for p, h in H.items() if h == KNOWN[k]] for k in KNOWN}
M = match(H)
if not all(M.values()):
    extra = extract_zips()
    print(f"Extracted/found {len(extra)} .pt files from zip archives")
    H.update({p: sha256(p) for p in extra}); M = match(H)
CH = {k: M[k][0] for k in KNOWN if M[k]}
MATCH = {k: True for k in CH}
if len(CH) < 2:
    print("\nDIAGNOSIS of every .pt file found:")
    for p, h in H.items(): describe(p, h)
    if ALLOW_UNMATCHED:
        for k in ("s", "m"):
            if k in CH: continue
            best = None
            for p in H:
                if p in CH.values(): continue
                r = info(p)
                if r is None or r[1] != NAMES or ("s" if r[0] < 14e6 else "m") != k: continue
                if best is None or ("last" not in p.name.lower()) > ("last" not in best.name.lower()): best = p
            if best: CH[k], MATCH[k] = best, False
    if len(CH) < 2:
        miss = [("YOLO11s" if k == "s" else "YOLO11m") for k in ("s", "m") if k not in CH]
        raise SystemExit(f"STOP: no checkpoint with the Mac SHA-256 for: {', '.join(miss)}. "
                         "Upload ~/Desktop/marine-detectors (two .pt files) as a Kaggle dataset, attach it under Input, and rerun. "
                         "Share this output if it still stops.")
for k, nm in (("s", "YOLO11s"), ("m", "YOLO11m")):
    r = info(CH[k])
    if r is None or r[1] != NAMES: raise SystemExit(f"STOP: {CH[k]} is not a 13-class BePLi checkpoint.")
    print(f"{nm}: {CH[k]} | params {r[0]:,} | SHA-256 {'MATCHES the Mac copy' if MATCH[k] else 'DIFFERS from the Mac copy'}")

# ---------- 4. BePLi YOLO dataset ----------
def count_ok():
    if not (YD / "bepli.yaml").exists(): return False
    for s, (ni, na) in EXPECTED.items():
        di, dl = YD / "images" / s, YD / "labels" / s
        if not (di.is_dir() and dl.is_dir()): return False
        if sum(1 for p in di.iterdir() if p.suffix.lower() in IMG_EXT) != ni: return False
        lfs = list(dl.glob("*.txt"))
        if len(lfs) != ni: return False
        if sum(sum(1 for l in f.read_text().splitlines() if l.strip()) for f in lfs) != na: return False
    return True

def find_root(base):
    for dp, dn, fn in os.walk(base):
        if os.path.basename(dp) == "plastic_coco" and os.path.isdir(os.path.join(dp, "annotation")) and os.path.isdir(os.path.join(dp, "images")):
            return Path(dp)
    return None

def convert(ROOT):
    shutil.rmtree(YD, ignore_errors=True)
    cmap = None
    for s, (ni, na) in EXPECTED.items():
        js = [q for q in sorted((ROOT / "annotation").glob("*.json")) if s in q.name.lower()]
        if not js: raise SystemExit(f"STOP: no annotation JSON for {s} in {ROOT / 'annotation'}")
        d = json.load(open(js[0]))
        if len(d["images"]) != ni or len(d["annotations"]) != na:
            raise SystemExit(f"STOP: {s} has {len(d['images'])} images / {len(d['annotations'])} annotations, expected {ni} / {na}")
        if cmap is None:
            cats = sorted(d["categories"], key=lambda c: c["id"])
            if [c["name"] for c in cats] != NAMES: raise SystemExit("STOP: class names or order differ from the trained models.")
            cmap = {c["id"]: i for i, c in enumerate(cats)}
        (YD / "images" / s).mkdir(parents=True, exist_ok=True); (YD / "labels" / s).mkdir(parents=True, exist_ok=True)
        by = {}
        for a in d["annotations"]: by.setdefault(a["image_id"], []).append(a)
        n_lines = 0
        for im in d["images"]:
            fn = Path(im["file_name"]).name
            src = ROOT / "images" / s / fn
            if not src.exists(): src = ROOT / "images" / s / im["file_name"]
            dst = YD / "images" / s / fn
            if not dst.exists():
                try: os.link(src, dst)
                except OSError: shutil.copy(src, dst)
            W, Hh = im["width"], im["height"]; lines = []
            for a in by.get(im["id"], []):
                x, y, w, h = a["bbox"]
                x0, y0, x1, y1 = max(0, x), max(0, y), min(W, x + w), min(Hh, y + h)
                if x1 <= x0 or y1 <= y0: continue
                lines.append(f"{cmap[a['category_id']]} {(x0+x1)/2/W:.6f} {(y0+y1)/2/Hh:.6f} {(x1-x0)/W:.6f} {(y1-y0)/Hh:.6f}")
            (YD / "labels" / s / (Path(fn).stem + ".txt")).write_text("\n".join(lines))
            n_lines += len(lines)
        if n_lines != na: raise SystemExit(f"STOP: {s} label lines {n_lines} != {na}")
        print(f"  converted {s}: images={ni} label_lines={n_lines}")
    (YD / "bepli.yaml").write_text(f"path: {YD}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES)))

if count_ok():
    print("YOLO dataset already present and verified in /tmp/yolo_bepli.")
else:
    ROOT = find_root(RAW) if OKFLAG.exists() else None
    if ROOT: print("Using extracted BePLi at", ROOT)
    if ROOT is None:
        ROOT = find_root("/kaggle/input")
        if ROOT: print("Using BePLi attached under /kaggle/input:", ROOT)
    if ROOT is None:
        BASE.mkdir(parents=True, exist_ok=True); RAW.mkdir(parents=True, exist_ok=True)
        free = shutil.disk_usage("/tmp").free / 1e9
        print(f"/tmp free: {free:.0f} GB")
        if free < 25: raise SystemExit("STOP: less than 25 GB free in /tmp.")
        if not ARCH.exists() or ARCH.stat().st_size < 7e9:
            print("Downloading BePLi v2 (about 9 GB, roughly 10-15 min) ...")
            rc = subprocess.call(["wget", "-q", "-c", "--tries=5", "--timeout=60", "-O", str(ARCH), URL])
            if rc != 0: raise SystemExit(f"STOP: download failed (wget exit {rc}). Session options -> Internet -> On, then rerun.")
        print(f"Archive {ARCH.stat().st_size/1e9:.2f} GB | testing integrity ...")
        if subprocess.call(["gzip", "-t", str(ARCH)]) != 0:
            raise SystemExit("STOP: archive corrupt. Delete /tmp/bepli/bepli.tar.gz and rerun.")
        print("Extracting ...")
        if subprocess.call(["tar", "-xzf", str(ARCH), "-C", str(RAW)]) != 0: raise SystemExit("STOP: extraction failed.")
        ROOT = find_root(RAW)
        if ROOT is None: raise SystemExit("STOP: plastic_coco folder not found after extraction.")
        OKFLAG.write_text("ok"); os.remove(ARCH)
    print("Converting COCO -> YOLO (official splits unchanged) ...")
    convert(ROOT)
if not count_ok(): raise SystemExit("STOP: YOLO dataset failed verification after conversion.")
sets = {s: {p.name for p in (YD / "images" / s).iterdir()} for s in EXPECTED}
ov = len(sets["train"] & sets["val"]) + len(sets["train"] & sets["test"]) + len(sets["val"] & sets["test"])
for s, (ni, na) in EXPECTED.items(): print(f"{s}: images={ni} label_lines={na} -> OK")
print("filename overlap across splits:", ov)
if ov: raise SystemExit("STOP: split overlap.")

json.dump({"S_PT": str(CH["s"]), "M_PT": str(CH["m"]), "S_SHA": H[CH["s"]], "M_SHA": H[CH["m"]], "MATCH": MATCH,
           "YAML": str(YD / "bepli.yaml"), "YOLO_DIR": str(YD), "GPU": gpu}, open(STATE, "w"), indent=1)
print(f"\nSETUP OK | {(time.time()-t0)/60:.1f} min | state: {STATE}")

In [ ]:
import os, sys, json, time, random, shutil, hashlib, zipfile, platform
from pathlib import Path
import numpy as np, torch, torch.nn as nn
from PIL import Image
import matplotlib.pyplot as plt
os.environ["WANDB_DISABLED"] = "true"
T0 = time.time()

# ================= preflight =================
STATE = Path("/kaggle/working/env_state.json")
if not STATE.exists(): raise SystemExit("STOP: /kaggle/working/env_state.json missing. Run the Setup cell first.")
E = json.load(open(STATE))
S_PT, M_PT, YAML, YD = Path(E["S_PT"]), Path(E["M_PT"]), Path(E["YAML"]), Path(E["YOLO_DIR"])
EXPECTED = {"train": (2234, 71889), "val": (744, 24643), "test": (744, 22040)}
IMG_EXT = {".png", ".jpg", ".jpeg"}

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

if not torch.cuda.is_available(): raise SystemExit("STOP: no CUDA GPU. Session options -> Accelerator -> GPU, then rerun.")
dev = "cuda"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)
for p in (S_PT, M_PT, YAML):
    if not p.exists(): raise SystemExit(f"STOP: missing {p}. Rerun the Setup cell.")
S_SHA, M_SHA = sha256(S_PT), sha256(M_PT)
if S_SHA != E["S_SHA"] or M_SHA != E["M_SHA"]: raise SystemExit("STOP: checkpoint hash differs from the Setup record. Rerun the Setup cell.")
for s, (ni, na) in EXPECTED.items():
    di, dl = YD / "images" / s, YD / "labels" / s
    n_img = sum(1 for p in di.iterdir() if p.suffix.lower() in IMG_EXT) if di.is_dir() else -1
    lfs = list(dl.glob("*.txt")) if dl.is_dir() else []
    n_lab = sum(sum(1 for l in f.read_text().splitlines() if l.strip()) for f in lfs)
    ok = n_img == ni and len(lfs) == ni and n_lab == na
    print(f"{s}: images={n_img} label_files={len(lfs)} label_lines={n_lab} -> {'OK' if ok else 'MISMATCH'}")
    if not ok: raise SystemExit("STOP: dataset mismatch. Rerun the Setup cell.")
print(f"YOLO11s SHA-256 {S_SHA[:12]} (matches Mac copy: {E['MATCH']['s']}) | YOLO11m SHA-256 {M_SHA[:12]} (matches Mac copy: {E['MATCH']['m']})")

CFG = dict(seed=0, crop=64, min_side=32, pad=1.3, train_cap=12000, val_cap=3000, vae_z=64, vae_beta=0.25,
           vae_epochs=40, vae_bs=128, gan_epochs=80, gan_bs=128, gan_nz=100, knn=5)
CK = hashlib.md5(json.dumps(CFG, sort_keys=True).encode()).hexdigest()[:8]
DK = S_SHA[:8] + "_" + M_SHA[:8]
CACHE = Path("/kaggle/working/gen_cache"); CACHE.mkdir(parents=True, exist_ok=True)
LOG = CACHE / "cell_e_log.txt"
def log(*a):
    s = " ".join(str(x) for x in a); print(s)
    with open(LOG, "a") as f: f.write(s + "\n")
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def show_png(path):
    plt.figure(figsize=(16, 5)); plt.imshow(plt.imread(str(path))); plt.axis("off"); plt.show()
log(f"\n===== Cell E run {time.strftime('%Y-%m-%d %H:%M:%S')} | config key {CK} | detector key {DK} =====")

# ================= Part 1: detector comparison (no training) =================
from ultralytics import YOLO
DET_F = CACHE / f"detector_comparison_{DK}.json"
DET = json.load(open(DET_F)) if DET_F.exists() else {}
for tag, w in [("YOLO11s", S_PT), ("YOLO11m", M_PT)]:
    need = [sp for sp in ("val", "test") if f"{tag}_{sp}" not in DET]
    if not need: continue
    mdl = YOLO(str(w))
    for sp in need:
        r = mdl.val(data=str(YAML), split=sp, imgsz=1280, batch=8, max_det=400, conf=0.001, iou=0.6, device=0, plots=False,
                    verbose=False, project=str(CACHE / "val_runs"), name=f"{tag}_{sp}", exist_ok=True)
        DET[f"{tag}_{sp}"] = {"P": float(r.box.mp), "R": float(r.box.mr), "mAP50": float(r.box.map50), "mAP50_95": float(r.box.map)}
        json.dump(DET, open(DET_F, "w"), indent=1)
    del mdl; torch.cuda.empty_cache()
log("\n=== DETECTOR COMPARISON (conf=0.001, iou=0.6, imgsz=1280, max_det=400) ===")
for tag in ("YOLO11s", "YOLO11m"):
    for sp in ("val", "test"):
        d = DET[f"{tag}_{sp}"]
        log(f"{tag} {sp}: P={d['P']:.3f} R={d['R']:.3f} mAP50={d['mAP50']:.3f} mAP50-95={d['mAP50_95']:.3f}")
for key, ref in (("YOLO11s_test", 0.144), ("YOLO11s_val", 0.160)):
    if abs(DET[key]["mAP50_95"] - ref) > 0.006:
        log(f"WARNING: {key} mAP50-95 {DET[key]['mAP50_95']:.3f} differs from the earlier run ({ref:.3f}). Check the checkpoint or dataset.")
    else:
        log(f"check: {key} mAP50-95 {DET[key]['mAP50_95']:.3f} agrees with the earlier run ({ref:.3f})")
log(f"[{(time.time()-T0)/60:.1f} min]")

# ================= Part 2: crops =================
def img_path(split, stem):
    for e in (".png", ".jpg", ".jpeg"):
        p = YD / "images" / split / (stem + e)
        if p.exists(): return p
    raise FileNotFoundError(f"{split}/{stem}")

def collect(split):
    meta = []
    for lf in sorted((YD / "labels" / split).glob("*.txt")):
        rows = [l.split() for l in lf.read_text().splitlines() if l.strip()]
        if not rows: continue
        with Image.open(img_path(split, lf.stem)) as im0: W, H = im0.size
        for q in rows:
            c = int(q[0]); cx, cy, bw, bh = [float(v) for v in q[1:5]]
            if max(bw * W, bh * H) >= CFG["min_side"]: meta.append((lf.stem, c, cx * W, cy * H, bw * W, bh * H, W, H))
    return meta

def build(meta, cap, seed, split):
    random.Random(seed).shuffle(meta); meta = sorted(meta[:cap])
    X, L, cur, im = [], [], None, None
    for stem, c, cx, cy, bw, bh, W, H in meta:
        if stem != cur:
            with Image.open(img_path(split, stem)) as im0: im = im0.convert("RGB")
            cur = stem
        s = min(max(bw, bh) * CFG["pad"], W, H)
        x0 = min(max(cx - s / 2, 0), W - s); y0 = min(max(cy - s / 2, 0), H - s)
        X.append(np.asarray(im.crop((round(x0), round(y0), round(x0 + s), round(y0 + s))).resize((CFG["crop"], CFG["crop"]), Image.LANCZOS)))
        L.append(c)
    return np.stack(X), np.array(L)

CR_F = CACHE / f"crops_{CK}.npz"
Xtr_u8 = None
if CR_F.exists():
    try:
        z = np.load(CR_F); Xtr_u8, Ltr, Xva_u8, Lva = z["Xtr"], z["Ltr"], z["Xva"], z["Lva"]
        assert Xtr_u8.shape[1:] == (64, 64, 3) and len(Xtr_u8) == len(Ltr) and len(Xva_u8) == len(Lva)
        log(f"crops loaded from cache: train {len(Xtr_u8)}, val {len(Xva_u8)}")
    except Exception as e:
        log("crop cache unusable, rebuilding:", str(e)[:100]); Xtr_u8 = None
if Xtr_u8 is None:
    Xtr_u8, Ltr = build(collect("train"), CFG["train_cap"], CFG["seed"], "train")
    Xva_u8, Lva = build(collect("val"), CFG["val_cap"], CFG["seed"] + 1, "val")
    np.savez(CR_F, Xtr=Xtr_u8, Ltr=Ltr, Xva=Xva_u8, Lva=Lva)
log(f"crops: train {len(Xtr_u8)}, val {len(Xva_u8)} | classes present in train {len(set(Ltr.tolist()))} | train class counts {np.bincount(Ltr, minlength=13).tolist()}")
tt = lambda a: torch.from_numpy(a).permute(0, 3, 1, 2).float().div(255.0).contiguous()
Xtr, Xva = tt(Xtr_u8), tt(Xva_u8)
Xtr_g, Xva_g = Xtr.to(dev), Xva.to(dev)
log(f"[{(time.time()-T0)/60:.1f} min]")

# ================= Part 3: VAE vs PCA =================
def cb(i, o): return nn.Sequential(nn.Conv2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.LeakyReLU(0.2))
def db(i, o): return nn.Sequential(nn.ConvTranspose2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.ReLU())
class VAE(nn.Module):
    def __init__(s, z=64):
        super().__init__()
        s.enc = nn.Sequential(cb(3, 32), cb(32, 64), cb(64, 128), cb(128, 256))
        s.mu = nn.Linear(4096, z); s.lv = nn.Linear(4096, z); s.fc = nn.Linear(z, 4096)
        s.dec = nn.Sequential(db(256, 128), db(128, 64), db(64, 32), nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid())
    def encode(s, x): h = s.enc(x).flatten(1); return s.mu(h), s.lv(h)
    def decode(s, z): return s.dec(s.fc(z).view(-1, 256, 4, 4))
    def forward(s, x):
        mu, lv = s.encode(x); z = mu + torch.randn_like(mu) * torch.exp(0.5 * lv)
        return s.decode(z), mu, lv

V_J, V_P, V_I, V_C = CACHE / f"vae_{CK}.json", CACHE / f"vae_{CK}.pt", CACHE / f"vae_recon_{CK}.png", CACHE / f"vae_ckpt_{CK}.pt"
if V_J.exists() and V_P.exists() and V_I.exists():
    VAEM = json.load(open(V_J)); log("VAE vs PCA: loaded from cache"); show_png(V_I)
else:
    def fresh():
        m = VAE(CFG["vae_z"]).to(dev); return m, torch.optim.Adam(m.parameters(), 1e-3)
    seed_all(CFG["seed"]); vae, opt = fresh(); start = 0
    if V_C.exists():
        try:
            ck = torch.load(V_C, map_location=dev); vae.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); start = int(ck["epoch"])
            log(f"VAE: resuming from epoch {start}")
        except Exception as e:
            log("VAE checkpoint unusable, restarting:", str(e)[:100]); vae, opt = fresh(); start = 0
    seed_all(CFG["seed"] + start)
    for ep in range(start, CFG["vae_epochs"]):
        vae.train(); perm = torch.randperm(len(Xtr_g), device=dev); tr = 0.0; nb = 0
        for i in range(0, len(perm), CFG["vae_bs"]):
            ix = perm[i:i + CFG["vae_bs"]]
            if len(ix) < 2: continue
            x = Xtr_g[ix]; xr, mu, lv = vae(x)
            rec = ((xr - x) ** 2).sum() / len(x); kl = (-0.5 * (1 + lv - mu ** 2 - lv.exp()).sum()) / len(x)
            loss = rec + CFG["vae_beta"] * kl; opt.zero_grad(); loss.backward(); opt.step(); tr += loss.item(); nb += 1
        if (ep + 1) % 10 == 0:
            log(f"  VAE epoch {ep+1}/{CFG['vae_epochs']} loss {tr/nb:.2f}")
            torch.save({"model": vae.state_dict(), "opt": opt.state_dict(), "epoch": ep + 1}, V_C)
    vae.eval()
    with torch.no_grad():
        Ztr = torch.cat([vae.encode(Xtr_g[i:i + 500])[0] for i in range(0, len(Xtr_g), 500)]).cpu().numpy()
        Zva_t = torch.cat([vae.encode(Xva_g[i:i + 500])[0] for i in range(0, len(Xva_g), 500)])
        rec_v = torch.cat([vae.decode(Zva_t[i:i + 500]) for i in range(0, len(Zva_t), 500)]).cpu()
    Zva = Zva_t.cpu().numpy(); mse_v = float(((rec_v - Xva) ** 2).mean())
    from sklearn.decomposition import PCA
    from sklearn.neighbors import KNeighborsClassifier
    ftr, fva = Xtr.reshape(len(Xtr), -1).numpy(), Xva.reshape(len(Xva), -1).numpy()
    pca = PCA(CFG["vae_z"], svd_solver="randomized", random_state=0).fit(ftr)
    Zp_tr, Zp_va = pca.transform(ftr), pca.transform(fva)
    rec_p = np.clip(pca.inverse_transform(Zp_va), 0, 1); mse_p = float(((rec_p - fva) ** 2).mean())
    knn = lambda a, b: float(KNeighborsClassifier(CFG["knn"]).fit(a, Ltr).score(b, Lva))
    acc_v, acc_p = knn(Ztr, Zva), knn(Zp_tr, Zp_va)
    maj = float((Lva == np.bincount(Ltr).argmax()).mean())
    VAEM = {"vae_mse": mse_v, "pca_mse": mse_p, "vae_knn_acc": acc_v, "pca_knn_acc": acc_p, "majority_train_class_on_val": maj,
            "latent_dim": CFG["vae_z"], "beta": CFG["vae_beta"], "epochs": CFG["vae_epochs"]}
    torch.save(vae.state_dict(), V_P)
    idx = np.linspace(0, len(Xva) - 1, 12).astype(int)
    fig, ax = plt.subplots(3, 12, figsize=(18, 5.2))
    for j, i in enumerate(idx):
        ax[0, j].imshow(Xva_u8[i]); ax[1, j].imshow(rec_v[i].permute(1, 2, 0).numpy())
        ax[2, j].imshow(rec_p[i].reshape(3, 64, 64).transpose(1, 2, 0))
        for r_ in range(3): ax[r_, j].axis("off")
    ax[0, 0].set_title("original", fontsize=9, loc="left"); ax[1, 0].set_title("VAE", fontsize=9, loc="left"); ax[2, 0].set_title("PCA-64", fontsize=9, loc="left")
    plt.tight_layout(); fig.savefig(V_I, dpi=110); plt.show(); plt.close(fig)
    json.dump(VAEM, open(V_J, "w"), indent=1)
log("\n=== VAE vs PCA (64 latent dims, held-out val crops, PCA output clipped to [0,1]) ===")
log(f"reconstruction MSE per pixel: VAE {VAEM['vae_mse']:.5f} | PCA {VAEM['pca_mse']:.5f}")
log(f"5-NN class accuracy in latent space (13 classes): VAE {VAEM['vae_knn_acc']:.3f} | PCA {VAEM['pca_knn_acc']:.3f} | train-majority-class baseline {VAEM['majority_train_class_on_val']:.3f}")
log(f"[{(time.time()-T0)/60:.1f} min]")

# ================= Part 4: DCGAN + copy check =================
class G(nn.Module):
    def __init__(s, nz=100):
        super().__init__()
        s.net = nn.Sequential(
            nn.ConvTranspose2d(nz, 512, 4, 1, 0, bias=False), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh())
    def forward(s, z): return s.net(z.view(z.size(0), -1, 1, 1))
class D(nn.Module):
    def __init__(s):
        super().__init__()
        s.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False), nn.BatchNorm2d(512), nn.LeakyReLU(0.2, True),
            nn.Conv2d(512, 1, 4, 1, 0))
    def forward(s, x): return s.net(x).view(-1)

G_J, G_P, G_I, G_C = CACHE / f"gan_{CK}.json", CACHE / f"gan_generator_{CK}.pt", CACHE / f"gan_samples_{CK}.png", CACHE / f"gan_ckpt_{CK}.pt"
NZ = CFG["gan_nz"]
if G_J.exists() and G_P.exists() and G_I.exists():
    GANM = json.load(open(G_J)); log("DCGAN: loaded from cache"); show_png(G_I)
else:
    def fresh_gan():
        g, d = G(NZ).to(dev), D().to(dev)
        return g, d, torch.optim.Adam(g.parameters(), 2e-4, betas=(0.5, 0.999)), torch.optim.Adam(d.parameters(), 2e-4, betas=(0.5, 0.999))
    seed_all(CFG["seed"] + 1); g, d, og, od = fresh_gan(); start = 0
    if G_C.exists():
        try:
            ck = torch.load(G_C, map_location=dev)
            g.load_state_dict(ck["g"]); d.load_state_dict(ck["d"]); og.load_state_dict(ck["og"]); od.load_state_dict(ck["od"]); start = int(ck["epoch"])
            log(f"DCGAN: resuming from epoch {start}")
        except Exception as e:
            log("GAN checkpoint unusable, restarting:", str(e)[:100]); g, d, og, od = fresh_gan(); start = 0
    seed_all(CFG["seed"] + 100 + start)
    Xg = Xtr_g * 2 - 1; bce = nn.BCEWithLogitsLoss()
    for ep in range(start, CFG["gan_epochs"]):
        g.train(); d.train(); perm = torch.randperm(len(Xg), device=dev); ld = lg = 0.0; nb = 0
        for i in range(0, len(perm), CFG["gan_bs"]):
            real = Xg[perm[i:i + CFG["gan_bs"]]]
            if len(real) < 2: continue
            if torch.rand(1).item() < 0.5: real = real.flip(3)
            z = torch.randn(len(real), NZ, device=dev); fake = g(z)
            l_d = bce(d(real), torch.full((len(real),), 0.9, device=dev)) + bce(d(fake.detach()), torch.zeros(len(real), device=dev))
            od.zero_grad(); l_d.backward(); od.step()
            l_g = bce(d(fake), torch.ones(len(real), device=dev)); og.zero_grad(); l_g.backward(); og.step()
            ld += l_d.item(); lg += l_g.item(); nb += 1
        if (ep + 1) % 10 == 0:
            if (ep + 1) % 20 == 0: log(f"  GAN epoch {ep+1}/{CFG['gan_epochs']} D loss {ld/nb:.3f} G loss {lg/nb:.3f}")
            torch.save({"g": g.state_dict(), "d": d.state_dict(), "og": og.state_dict(), "od": od.state_dict(), "epoch": ep + 1}, G_C)
    g.eval()
    gen_t = torch.Generator(device=dev); gen_t.manual_seed(123)
    def nn_dist(a, b):
        return torch.cat([torch.cdist(a[i:i + 64], b).min(1).values for i in range(0, len(a), 64)])
    with torch.no_grad():
        samples = ((g(torch.randn(64, NZ, device=dev, generator=gen_t)) + 1) / 2).clamp(0, 1).cpu()
        gen = g(torch.randn(256, NZ, device=dev, generator=gen_t)).flatten(1)
        vi = np.random.default_rng(7).choice(len(Xva_g), min(256, len(Xva_g)), replace=False)
        va = (Xva_g[vi] * 2 - 1).flatten(1); tr = Xg.flatten(1)
        d_gen, d_val = nn_dist(gen, tr), nn_dist(va, tr)
    GANM = {"gen_median": float(d_gen.median()), "gen_min": float(d_gen.min()), "val_median": float(d_val.median()), "val_min": float(d_val.min()),
            "frac_gen_closer_than_every_val": float((d_gen < d_val.min()).float().mean()), "epochs": CFG["gan_epochs"], "n_gen": int(len(d_gen)), "n_val": int(len(d_val))}
    torch.save(g.state_dict(), G_P)
    grid = samples.reshape(8, 8, 3, 64, 64).permute(0, 3, 1, 4, 2).reshape(512, 512, 3).numpy()
    fig = plt.figure(figsize=(8, 8)); plt.imshow(grid); plt.axis("off"); plt.title("DCGAN samples (64x64 crops of annotated litter)")
    fig.savefig(G_I, dpi=110, bbox_inches="tight"); plt.show(); plt.close(fig)
    json.dump(GANM, open(G_J, "w"), indent=1)
log("\n=== GAN copy check (pixel-space L2 to nearest training crop, scale [-1,1]) ===")
log(f"generated ({GANM['n_gen']}): median {GANM['gen_median']:.1f}, min {GANM['gen_min']:.1f} | real held-out val crops ({GANM['n_val']}): median {GANM['val_median']:.1f}, min {GANM['val_min']:.1f}")
log(f"share of generated samples closer to a training crop than every real val crop: {GANM['frac_gen_closer_than_every_val']*100:.1f}% (near 0% means no sign of memorisation)")
log("No FID computed. Sample quality is judged visually from the grid only.")
log(f"[{(time.time()-T0)/60:.1f} min]")

# ================= Part 5: bundle =================
import sklearn, ultralytics
st = Path("/kaggle/working/gen_stage"); shutil.rmtree(st, ignore_errors=True); st.mkdir(parents=True)
for src, dst in [(V_P, "vae.pt"), (G_P, "gan_generator.pt"), (V_I, "vae_recon.png"), (G_I, "gan_samples.png"), (DET_F, "detector_comparison.json"), (LOG, "cell_e_log.txt")]:
    shutil.copy(src, st / dst)
GM = {"config": CFG, "train_crops": int(len(Xtr_u8)), "val_crops": int(len(Xva_u8)), "detectors": DET, "vae_vs_pca": VAEM, "gan_copy_check": GANM,
      "checkpoints": {"yolo11s": {"sha256": S_SHA, "identical_to_mac_copy": E["MATCH"]["s"]}, "yolo11m": {"sha256": M_SHA, "identical_to_mac_copy": E["MATCH"]["m"]}}}
json.dump(GM, open(st / "generative_metrics.json", "w"), indent=1)
json.dump({"python": platform.python_version(), "torch": torch.__version__, "ultralytics": ultralytics.__version__, "sklearn": sklearn.__version__,
           "gpu": torch.cuda.get_device_name(0), "seed": CFG["seed"], "finished": time.strftime("%Y-%m-%d %H:%M:%S"),
           "note": "GAN/VAE resumed runs are not bit-identical to uninterrupted runs"}, open(st / "run_info.json", "w"), indent=1)
Z = shutil.make_archive("/kaggle/working/generative_bundle", "zip", str(st))
with zipfile.ZipFile(Z) as zf:
    assert zf.testzip() is None, "zip corrupt"
    names = zf.namelist()
print(f"\nBundle: {Z} ({os.path.getsize(Z)/1e6:.1f} MB) | files: {names}")
print(f"Total this session: {(time.time()-T0)/60:.1f} min")
print("DONE-E. Download generative_bundle.zip from the Output panel (/kaggle/working) now.")